In [41]:
import optuna
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [14]:
# Our objective function
def objective(trial):

    # Picks the float for x in the given range but using TPESampler internally
    x = trial.suggest_float("x", 0, 6)

    # Returns the score
    return -(x - 3) ** 2 + 10

# Create the study object and pass the direction (whether to maximize or minimize the values)
study = optuna.create_study(direction="maximize")
# Pass our objective function and the number of times the trials must be runned
study.optimize(objective, n_trials=20)

[I 2026-07-26 16:22:27,626] A new study created in memory with name: no-name-7b225153-2234-4b4a-9244-97ca1543cfd6
[I 2026-07-26 16:22:27,630] Trial 0 finished with value: 9.833344707099144 and parameters: {'x': 3.408234360264855}. Best is trial 0 with value: 9.833344707099144.
[I 2026-07-26 16:22:27,633] Trial 1 finished with value: 9.710104626355026 and parameters: {'x': 3.5384193288181374}. Best is trial 0 with value: 9.833344707099144.
[I 2026-07-26 16:22:27,635] Trial 2 finished with value: 9.83060157887958 and parameters: {'x': 3.4115803944801297}. Best is trial 0 with value: 9.833344707099144.
[I 2026-07-26 16:22:27,638] Trial 3 finished with value: 9.337407609288386 and parameters: {'x': 3.8139977829893734}. Best is trial 0 with value: 9.833344707099144.
[I 2026-07-26 16:22:27,642] Trial 4 finished with value: 1.0858222457046711 and parameters: {'x': 5.985662029482796}. Best is trial 0 with value: 9.833344707099144.
[I 2026-07-26 16:22:27,644] Trial 5 finished with value: 3.5195

In [15]:
study.best_params,study.best_value # We can fetch the best params and best value

({'x': 2.9298392824040613}, 9.995077473706424)

In [ ]:
print(study.best_trial) # The metadata of the best trial 11 as seen from above logs
# Optuna stores the entire trial object it stores : 
""" 
trial number
start time
end time
duration
parameter values
distributions
user attributes
system attributes
intermediate values (for pruning)
state (completed, pruned, failed)
"""

FrozenTrial(number=7, state=<TrialState.COMPLETE: 1>, values=[9.995077473706424], datetime_start=datetime.datetime(2026, 7, 26, 16, 22, 27, 648554), datetime_complete=datetime.datetime(2026, 7, 26, 16, 22, 27, 649031), params={'x': 2.9298392824040613}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'x': FloatDistribution(high=6.0, log=False, low=0.0, step=None)}, trial_id=7, value=None)


In [ ]:
# A responsible worflow and code base looks like : 
""" 
Choose Metric
        │
        ▼
Choose Search Space
        │
        ▼
Choose Validation Strategy
        │
        ▼
Run Optuna
        │
        ▼
Train Final Model
        │
        ▼
Evaluate Once on Test Set
"""

# Also use cross validation splitting inside the objective function to make sure we are actually training on training set and testing parameters on val set.
# The test set is always locked away till the end.

In [21]:
# Now we implement optuna on a standard breast cancer dataset.

data = load_breast_cancer()

X = data.data
y = data.target

In [25]:
X[:2]

array([[1.799e+01, 1.038e+01, 1.228e+02, 1.001e+03, 1.184e-01, 2.776e-01,
        3.001e-01, 1.471e-01, 2.419e-01, 7.871e-02, 1.095e+00, 9.053e-01,
        8.589e+00, 1.534e+02, 6.399e-03, 4.904e-02, 5.373e-02, 1.587e-02,
        3.003e-02, 6.193e-03, 2.538e+01, 1.733e+01, 1.846e+02, 2.019e+03,
        1.622e-01, 6.656e-01, 7.119e-01, 2.654e-01, 4.601e-01, 1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, 1.326e+03, 8.474e-02, 7.864e-02,
        8.690e-02, 7.017e-02, 1.812e-01, 5.667e-02, 5.435e-01, 7.339e-01,
        3.398e+00, 7.408e+01, 5.225e-03, 1.308e-02, 1.860e-02, 1.340e-02,
        1.389e-02, 3.532e-03, 2.499e+01, 2.341e+01, 1.588e+02, 1.956e+03,
        1.238e-01, 1.866e-01, 2.416e-01, 1.860e-01, 2.750e-01, 8.902e-02]])

In [26]:
y[:2]

array([0, 0])

In [ ]:
# Split the dataset into training and testing set.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y # To make sure that in each split we have reasonable ratio of y maintained (Cause the output is imbalanced )
)

In [37]:
# Build the objective function : 
def objective(trial):

    max_depth = trial.suggest_int(
        "max_depth",
        2,
        10
    )

    n_estimators = trial.suggest_int(
        "n_estimators",
        50,
        150
    )

    min_samples_split = trial.suggest_int(
        "min_samples_split",
        2,
        20
    )

    model = RandomForestClassifier(
        max_depth=max_depth,
        n_estimators=n_estimators,
        min_samples_split=min_samples_split,
        random_state=42,
        n_jobs=-1
    )

    # Pass the model,training set and the no of splits to predict a generalized score
    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    ).mean()

    return score

In [38]:
# Now creating the study object and passing the objective function : 
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=25
)

[I 2026-07-26 16:42:09,917] A new study created in memory with name: no-name-a25c7900-a2be-4068-9a05-dac8dc4edc0b
[I 2026-07-26 16:42:25,258] Trial 0 finished with value: 0.9472527472527472 and parameters: {'max_depth': 9, 'n_estimators': 68, 'min_samples_split': 19}. Best is trial 0 with value: 0.9472527472527472.
[I 2026-07-26 16:42:27,671] Trial 1 finished with value: 0.9516483516483518 and parameters: {'max_depth': 4, 'n_estimators': 146, 'min_samples_split': 18}. Best is trial 1 with value: 0.9516483516483518.
[I 2026-07-26 16:42:28,902] Trial 2 finished with value: 0.9472527472527472 and parameters: {'max_depth': 2, 'n_estimators': 53, 'min_samples_split': 19}. Best is trial 1 with value: 0.9516483516483518.
[I 2026-07-26 16:42:30,288] Trial 3 finished with value: 0.9516483516483518 and parameters: {'max_depth': 8, 'n_estimators': 107, 'min_samples_split': 12}. Best is trial 1 with value: 0.9516483516483518.
[I 2026-07-26 16:42:31,820] Trial 4 finished with value: 0.9538461538461

In [39]:
print(study.best_params)

{'max_depth': 9, 'n_estimators': 141, 'min_samples_split': 4}


In [42]:
# Fetch the best model and predict using the test set

best_model = RandomForestClassifier(
    **study.best_params,
    random_state=42
)

best_model.fit(X_train, y_train)

pred = best_model.predict(X_test)

print(
    accuracy_score(
        y_test,
        pred
    )
)

0.9473684210526315
